In [ ]:
from pathlib import Path
import pandas as pd

mid_folder = Path("data/raw/military/mid")
mid_path = mid_folder / "dyadic_mid_4.03.csv"

assert mid_path.exists(), "The MID CSV file was not found."

mid_raw = pd.read_csv(mid_path)

print("Dataset shape:", mid_raw.shape)
print("Columns:", mid_raw.columns.tolist())

display(mid_raw.head())

In [ ]:
CHINA_CODE = 710
START_YEAR = 1950
END_YEAR = 2014

china_mid = mid_raw.loc[
    (mid_raw["statea"] == CHINA_CODE)
    & mid_raw["year"].between(START_YEAR, END_YEAR)
].copy()

print("China-directed dyad-year records:", len(china_mid))
print("China labels:", china_mid["namea"].unique())
print("Observed year range:",
      china_mid["year"].min(),
      "to",
      china_mid["year"].max())

display(
    china_mid[
        [
            "disno", "dyindex", "year",
            "statea", "namea", "stateb", "nameb",
            "hihost", "hihosta", "hihostb",
            "dyad_rolea", "war", "durindx"
        ]
    ].head(10)
)

In [ ]:
key_columns = ["dyindex", "year", "statea", "stateb"]

duplicate_mask = china_mid.duplicated(
    subset=key_columns,
    keep=False
)

print("Duplicate-key records:", duplicate_mask.sum())

assert not duplicate_mask.any(), (
    "Duplicate keys found. Inspect them before aggregation."
)

print("Unique dyadic disputes:", china_mid["dyindex"].nunique())
print("Unique parent disputes:", china_mid["disno"].nunique())
print("Distinct counterpart codes:", china_mid["stateb"].nunique())

In [ ]:
print("China-specific hostility levels:")
print(
    china_mid["hihosta"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nChina's dyadic roles:")
print(
    china_mid["dyad_rolea"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
interim_folder = Path("data/interim/military")
interim_folder.mkdir(parents=True, exist_ok=True)

china_mid.to_csv(
    interim_folder / "china_directed_dyad_year_1950_2014.csv",
    index=False,
    encoding="utf-8-sig"
)

print("China MID subset saved.")

In [ ]:
# Count distinct active disputes and counterparts in each year.
annual_active = (
    china_mid.groupby("year")
    .agg(
        mid_active_dispute_count=("disno", "nunique"),
        mid_active_dyad_count=("dyindex", "nunique"),
        mid_counterpart_count=("stateb", "nunique")
    )
    .reset_index()
)

# The codebook identifies the first dyadic-dispute year as durindx == 1.
onset_records = china_mid.loc[
    china_mid["durindx"] == 1
].copy()

annual_onset = (
    onset_records.groupby("year")["dyindex"]
    .nunique()
    .rename("mid_dyadic_onset_count")
    .reset_index()
)

display(annual_active.head())
display(annual_onset.head())

In [ ]:
annual_military = pd.DataFrame({
    "year": range(START_YEAR, END_YEAR + 1)
})

annual_military = (
    annual_military
    .merge(
        annual_active,
        on="year",
        how="left",
        validate="one_to_one"
    )
    .merge(
        annual_onset,
        on="year",
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "mid_active_dispute_count",
    "mid_active_dyad_count",
    "mid_counterpart_count",
    "mid_dyadic_onset_count"
]

annual_military[count_columns] = (
    annual_military[count_columns]
    .fillna(0)
    .astype(int)
)

annual_military["mid_active_any"] = (
    annual_military["mid_active_dispute_count"] > 0
).astype(int)

annual_military["mid_dyadic_onset_any"] = (
    annual_military["mid_dyadic_onset_count"] > 0
).astype(int)

annual_military["source_version"] = "Dyadic MID 4.03"

display(annual_military.head(15))

In [ ]:
print("Active-dispute outcome:")
print(
    annual_military["mid_active_any"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

print("\nDyadic-onset outcome:")
print(
    annual_military["mid_dyadic_onset_any"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

print("\nAnnual count summary:")
display(
    annual_military[count_columns].describe()
)

In [ ]:
processed_folder = Path("data/processed/military")
processed_folder.mkdir(parents=True, exist_ok=True)

annual_military.to_csv(
    processed_folder / "annual_mid_candidates_1950_2014.csv",
    index=False,
    encoding="utf-8-sig"
)

assert annual_military["year"].is_unique
assert len(annual_military) == 65

print("Saved 65 annual observations for 1950–2014.")

In [ ]:
# Use the dispute index and both participants together.
china_mid["dyad_dispute_id"] = (
    china_mid["dyindex"].astype("string")
    + "::"
    + china_mid["statea"].astype("string")
    + "::"
    + china_mid["stateb"].astype("string")
)

assert not china_mid.duplicated(
    ["dyad_dispute_id", "year"]
).any()

print(
    "Unique dyadic disputes:",
    china_mid["dyad_dispute_id"].nunique()
)

In [ ]:
annual_active = (
    china_mid.groupby("year")
    .agg(
        mid_active_dispute_count=("disno", "nunique"),
        mid_active_dyad_count=("dyad_dispute_id", "nunique"),
        mid_counterpart_count=("stateb", "nunique")
    )
    .reset_index()
)

onset_records = china_mid.loc[
    china_mid["durindx"] == 1
].copy()

annual_onset = (
    onset_records.groupby("year")["dyad_dispute_id"]
    .nunique()
    .rename("mid_dyadic_onset_count")
    .reset_index()
)

annual_military = pd.DataFrame({
    "year": range(START_YEAR, END_YEAR + 1)
})

annual_military = (
    annual_military
    .merge(
        annual_active,
        on="year",
        how="left",
        validate="one_to_one"
    )
    .merge(
        annual_onset,
        on="year",
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "mid_active_dispute_count",
    "mid_active_dyad_count",
    "mid_counterpart_count",
    "mid_dyadic_onset_count"
]

annual_military[count_columns] = (
    annual_military[count_columns]
    .fillna(0)
    .astype(int)
)

annual_military["mid_active_any"] = (
    annual_military["mid_active_dispute_count"] > 0
).astype(int)

annual_military["mid_dyadic_onset_any"] = (
    annual_military["mid_dyadic_onset_count"] > 0
).astype(int)

annual_military["source_version"] = "Dyadic MID 4.03"

# Check the aggregation logic.
assert (
    annual_military["mid_active_dyad_count"]
    >= annual_military["mid_counterpart_count"]
).all()

assert (
    annual_military["mid_dyadic_onset_count"]
    <= annual_military["mid_active_dyad_count"]
).all()

display(annual_military[count_columns].describe())

In [ ]:
for column in ["mid_active_any", "mid_dyadic_onset_any"]:
    print(column)
    print(
        annual_military[column]
        .value_counts()
        .reindex([0, 1], fill_value=0)
    )
    print()

annual_military.to_csv(
    processed_folder / "annual_mid_candidates_1950_2014.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Corrected annual outcomes saved.")

In [ ]:
print("War indicator values:")
print(
    china_mid["war"]
    .value_counts(dropna=False)
    .sort_index()
)

assert china_mid["war"].notna().all(), (
    "Missing war indicators need review."
)

assert china_mid["war"].isin([0, 1]).all(), (
    "Unexpected war indicator values need review."
)

In [ ]:
annual_war = (
    china_mid.groupby("year")["war"]
    .max()
    .rename("mid_war_any")
    .reset_index()
)

annual_military = annual_military.merge(
    annual_war,
    on="year",
    how="left",
    validate="one_to_one"
)

# In covered years with no MID records, there is no recorded MID war.
annual_military["mid_war_any"] = (
    annual_military["mid_war_any"]
    .fillna(0)
    .astype(int)
)

print("Annual war outcome:")
print(
    annual_military["mid_war_any"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

print("\nYears coded as war:")
print(
    annual_military.loc[
        annual_military["mid_war_any"] == 1,
        "year"
    ].tolist()
)

In [ ]:
war_records = china_mid.loc[
    china_mid["war"] == 1,
    [
        "disno",
        "dyad_dispute_id",
        "year",
        "nameb",
        "war",
        "hihosta"
    ]
].sort_values(["year", "nameb"])

display(war_records)

In [ ]:
required_columns = [
    "year",
    "mid_active_dispute_count"
]

assert all(
    column in annual_military.columns
    for column in required_columns
), "Required columns are missing."

assert annual_military["year"].is_unique
assert len(annual_military) == 65

assert annual_military[
    "mid_active_dispute_count"
].notna().all()

assert (
    annual_military["mid_active_dispute_count"] >= 0
).all()

assert (
    annual_military["mid_active_dispute_count"] % 1 == 0
).all()

print("Annual outcome checks passed.")

In [ ]:
military_target = annual_military[
    ["year", "mid_active_dispute_count"]
].copy()

military_target = military_target.rename(columns={
    "mid_active_dispute_count": "y_active_mid_count"
})

military_target["y_active_mid_count"] = (
    military_target["y_active_mid_count"].astype(int)
)

military_target = (
    military_target
    .sort_values("year")
    .reset_index(drop=True)
)

print("Target observations:", len(military_target))
display(military_target.head(15))

In [ ]:
expected_counts = (
    china_mid.groupby("year")["disno"]
    .nunique()
    .reindex(military_target["year"], fill_value=0)
    .to_numpy()
)

assert (
    military_target["y_active_mid_count"].to_numpy()
    == expected_counts
).all()

print("Target values match the underlying dispute records.")

In [ ]:
processed_folder = Path("data/processed/military")
processed_folder.mkdir(parents=True, exist_ok=True)

military_target.to_csv(
    processed_folder / "annual_military_target.csv",
    index=False,
    encoding="utf-8-sig"
)

annual_military.to_csv(
    processed_folder / "annual_mid_candidates_1950_2014.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Main target saved:")
print(
    (processed_folder / "annual_military_target.csv").resolve()
)

In [ ]:
import json

target_definition = {
    "target_name": "y_active_mid_count",
    "task": "count regression",
    "unit_of_analysis": "calendar year",
    "definition": (
        "Number of distinct parent MID dispute IDs involving "
        "China that are active in each year."
    ),
    "source": "Dyadic MID 4.03",
    "china_country_code": 710,
    "filter": "statea == 710",
    "year_start": 1950,
    "year_end": 2014,
    "aggregation": "Number of unique disno values per year",
    "zero_definition": (
        "No qualifying dispute records in a covered year."
    ),
    "planned_prediction": "Poster features in year t predict Y in year t+1",
    "limitations": [
        "Measures dispute participation, not necessarily initiation.",
        "Does not measure war counts or military severity.",
        "A continuing dispute contributes to each active year.",
        "Years outside source coverage must not be filled with zero."
    ]
}

target_definition_path = (
    processed_folder / "military_target_definition.json"
)

target_definition_path.write_text(
    json.dumps(target_definition, indent=2),
    encoding="utf-8"
)

print("Target definition saved.")